# `Dynamic AI Knowledge Assistant`

In [15]:
# ======== Imports ==========

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer




sys.path.append(str(Path.cwd().parent))

In [16]:
# ============ LOAD PHASE 7 RETRIEVAL PIPELINE =========================

# ============================================================
# STEP 1 - FIND PROJECT ROOT
# ============================================================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

    while PROJECT_ROOT != PROJECT_ROOT.parent:

        if (PROJECT_ROOT / "src").exists():
            break

        PROJECT_ROOT = PROJECT_ROOT.parent


PROJECT_ROOT = PROJECT_ROOT.resolve()


# ============================================================
# STEP 2 - ADD PROJECT ROOT TO PYTHON PATH
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# STEP 3 - DEFINE PHASE 7 PATHS
# ============================================================

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

CHUNK_METADATA_DIR = (
    PROCESSED_DIR
    / "chunk_metadata"
)

EMBEDDINGS_DIR = (
    PROCESSED_DIR
    / "embeddings"
)

VECTOR_DB_DIR = (
    PROCESSED_DIR
    / "vector_db"
)

CHROMA_DB_DIR = (
    VECTOR_DB_DIR
    / "chroma"
)

CHUNK_METADATA_PATH = (
    CHUNK_METADATA_DIR
    / "chunk_metadata.csv"
)

EMBEDDINGS_PATH = (
    EMBEDDINGS_DIR
    / "all_MiniLM_L6_v2_embeddings.npy"
)


# ============================================================
# STEP 4 - DISPLAY CONFIGURATION
# ============================================================

print("=" * 70)
print("PHASE 9 - LOAD PHASE 7 RETRIEVAL PIPELINE")
print("=" * 70)

print("Project root       :", PROJECT_ROOT)
print("Chunk metadata     :", CHUNK_METADATA_PATH)
print("Embeddings         :", EMBEDDINGS_PATH)
print("ChromaDB           :", CHROMA_DB_DIR)


# ============================================================
# STEP 5 - VERIFY PHASE 7 ARTIFACTS
# ============================================================

required_paths = {
    "Project root": PROJECT_ROOT,
    "Chunk metadata": CHUNK_METADATA_PATH,
    "Embeddings": EMBEDDINGS_PATH,
    "ChromaDB": CHROMA_DB_DIR
}

print("\n" + "=" * 70)
print("PHASE 7 ARTIFACT VERIFICATION")
print("=" * 70)

for name, path in required_paths.items():

    exists = path.exists()

    print(f"{name:<20}: {exists}")

    if not exists:
        raise FileNotFoundError(
            f"Required Phase 7 artifact not found: {path}"
        )


# ============================================================
# STEP 6 - LOAD CHUNK METADATA
# ============================================================

chunk_metadata_df = pd.read_csv(
    CHUNK_METADATA_PATH
)

print("\n" + "=" * 70)
print("CHUNK METADATA")
print("=" * 70)

print("Shape:", chunk_metadata_df.shape)

print(
    "Columns:",
    chunk_metadata_df.columns.tolist()
)


# ============================================================
# STEP 7 - VALIDATE CHUNK METADATA
# ============================================================

EXPECTED_COLUMNS = [
    "chunk_id",
    "article_id",
    "title",
    "category",
    "source_url",
    "original_file_name",
    "chunk_index",
    "chunk_text",
    "chunk_length"
]

missing_columns = [
    column
    for column in EXPECTED_COLUMNS
    if column not in chunk_metadata_df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing metadata columns: {missing_columns}"
    )


if chunk_metadata_df["chunk_id"].is_unique is False:

    raise ValueError(
        "chunk_id values are not unique."
    )


if chunk_metadata_df["chunk_text"].isna().any():

    raise ValueError(
        "Some chunk_text values are missing."
    )


print("Metadata validation : PASSED")


# ============================================================
# STEP 8 - LOAD EMBEDDINGS
# ============================================================

embeddings = np.load(
    EMBEDDINGS_PATH
)

print("\n" + "=" * 70)
print("EMBEDDINGS")
print("=" * 70)

print("Shape :", embeddings.shape)
print("Dtype :", embeddings.dtype)


if embeddings.shape[0] != len(chunk_metadata_df):

    raise ValueError(
        "Embedding count does not match chunk metadata count."
    )


if embeddings.shape[1] != 384:

    raise ValueError(
        f"Expected embedding dimension 384, "
        f"but found {embeddings.shape[1]}."
    )


print("Embedding validation : PASSED")


# ============================================================
# STEP 9 - CONNECT TO EXISTING CHROMADB
# ============================================================

print("\n" + "=" * 70)
print("CHROMADB")
print("=" * 70)

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DB_DIR)
)


# ============================================================
# STEP 10 - LOAD EXISTING COLLECTION
# ============================================================

COLLECTION_NAME = "wiki_chunks_minilm"

collection = chroma_client.get_collection(
    name=COLLECTION_NAME
)

collection_count = collection.count()

print("Collection :", COLLECTION_NAME)
print("Documents  :", collection_count)


if collection_count == 0:

    raise ValueError(
        "ChromaDB collection is empty."
    )


print("ChromaDB validation : PASSED")


# ============================================================
# STEP 11 - LOAD SAME EMBEDDING MODEL USED IN PHASE 7
# ============================================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("\n" + "=" * 70)
print("EMBEDDING MODEL")
print("=" * 70)

print(
    "Loading:",
    EMBEDDING_MODEL_NAME
)

model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

embedding_dimension = (
    model.get_embedding_dimension()
)

print(
    "Embedding dimension:",
    embedding_dimension
)


if embedding_dimension != 384:

    raise ValueError(
        "Embedding model dimension does not match "
        "Phase 7 embeddings."
    )


print("Embedding model validation : PASSED")


# ============================================================
# STEP 12 - QUERY EMBEDDING FUNCTION
# ============================================================

def generate_query_embedding(query):
    """
    Convert a user query into a normalized
    384-dimensional embedding.

    This is the same embedding approach
    used in Phase 7.
    """

    if not isinstance(query, str):

        raise TypeError(
            "Query must be a string."
        )

    query = query.strip()

    if not query:

        raise ValueError(
            "Query cannot be empty."
        )

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    query_embedding = (
        query_embedding
        .astype(np.float32)
    )

    return query_embedding


# ============================================================
# STEP 13 - BASIC SIMILARITY SEARCH
# ============================================================

def basic_similarity_search(
    query,
    top_k=5
):
    """
    Retrieve the top-K semantically similar
    chunks from the existing ChromaDB collection.

    Returns:
        pandas.DataFrame
    """

    # --------------------------------------------------------
    # Validate query
    # --------------------------------------------------------

    if not isinstance(query, str):

        raise TypeError(
            "Query must be a string."
        )

    query = query.strip()

    if not query:

        raise ValueError(
            "Query cannot be empty."
        )

    # --------------------------------------------------------
    # Validate top_k
    # --------------------------------------------------------

    if not isinstance(top_k, int):

        raise TypeError(
            "top_k must be an integer."
        )

    if top_k <= 0:

        raise ValueError(
            "top_k must be greater than zero."
        )

    if top_k > collection_count:

        raise ValueError(
            f"top_k cannot exceed collection size "
            f"({collection_count})."
        )

    # --------------------------------------------------------
    # Generate query embedding
    # --------------------------------------------------------

    query_embedding = (
        generate_query_embedding(query)
    )

    # --------------------------------------------------------
    # Query ChromaDB
    # --------------------------------------------------------

    results = collection.query(

        query_embeddings=(
            query_embedding.tolist()
        ),

        n_results=top_k,

        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    # --------------------------------------------------------
    # Extract results
    # --------------------------------------------------------

    ids = results["ids"][0]

    documents = results["documents"][0]

    metadatas = results["metadatas"][0]

    distances = results["distances"][0]

    # --------------------------------------------------------
    # Convert cosine distance
    # to cosine similarity
    # --------------------------------------------------------

    similarities = [
        1 - distance
        for distance in distances
    ]

    # --------------------------------------------------------
    # Build retrieval records
    # --------------------------------------------------------

    records = []

    for rank in range(len(ids)):

        metadata = metadatas[rank]

        record = {

            "rank":
                rank + 1,

            "chunk_id":
                ids[rank],

            "article_id":
                metadata["article_id"],

            "title":
                metadata["title"],

            "category":
                metadata["category"],

            "source_url":
                metadata["source_url"],

            "original_file_name":
                metadata["original_file_name"],

            "chunk_index":
                metadata["chunk_index"],

            "chunk_text":
                documents[rank],

            "similarity_score":
                similarities[rank]
        }

        records.append(record)


    # --------------------------------------------------------
    # Convert to DataFrame
    # --------------------------------------------------------

    results_df = pd.DataFrame(
        records
    )

    return results_df


# ============================================================
# STEP 14 - TEST RETRIEVER
# ============================================================

TEST_QUERY = (
    "What is artificial intelligence?"
)

test_results_df = basic_similarity_search(
    query=TEST_QUERY,
    top_k=5
)


# ============================================================
# STEP 15 - DISPLAY TEST RESULTS
# ============================================================

print("\n" + "=" * 70)
print("PHASE 7 RETRIEVER TEST")
print("=" * 70)

print("Query:")
print(TEST_QUERY)

print(
    "\nRetrieved chunks:",
    len(test_results_df)
)

display(
    test_results_df[
        [
            "rank",
            "chunk_id",
            "title",
            "category",
            "chunk_index",
            "similarity_score",
            "source_url"
        ]
    ]
)


# ============================================================
# STEP 16 - VALIDATE RETRIEVAL OUTPUT
# ============================================================

REQUIRED_RESULT_COLUMNS = {

    "rank",
    "chunk_id",
    "article_id",
    "title",
    "category",
    "source_url",
    "original_file_name",
    "chunk_index",
    "chunk_text",
    "similarity_score"
}


missing_result_columns = (
    REQUIRED_RESULT_COLUMNS
    - set(test_results_df.columns)
)

if missing_result_columns:

    raise ValueError(
        "Missing retrieval result columns: "
        f"{missing_result_columns}"
    )


if test_results_df["chunk_id"].is_unique is False:

    raise ValueError(
        "Retrieved chunk IDs are not unique."
    )


if test_results_df["chunk_text"].isna().any():

    raise ValueError(
        "Retrieved chunk text contains missing values."
    )


if test_results_df["source_url"].isna().any():

    raise ValueError(
        "Retrieved source URLs contain missing values."
    )


if not test_results_df[
    "similarity_score"
].between(-1, 1).all():

    raise ValueError(
        "Similarity scores must be between -1 and 1."
    )


if not test_results_df[
    "similarity_score"
].is_monotonic_decreasing:

    raise ValueError(
        "Similarity scores are not correctly ranked."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

print("Project configuration : PASSED")
print("Chunk metadata        : PASSED")
print("Embeddings            : PASSED")
print("ChromaDB              : PASSED")
print("Embedding model       : PASSED")
print("Query embedding       : PASSED")
print("Similarity search     : PASSED")
print("Retrieval output      : PASSED")

print("\nPhase 7 retrieval pipeline successfully loaded.")


PHASE 9 - LOAD PHASE 7 RETRIEVAL PIPELINE
Project root       : C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant
Chunk metadata     : C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\chunk_metadata\chunk_metadata.csv
Embeddings         : C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\embeddings\all_MiniLM_L6_v2_embeddings.npy
ChromaDB           : C:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\data\processed\vector_db\chroma

PHASE 7 ARTIFACT VERIFICATION
Project root        : True
Chunk metadata      : True
Embeddings          : True
ChromaDB            : True

CHUNK METADATA
Shape: (4693, 9)
Columns: ['chunk_id', 'article_id', 'title', 'category', 'source_url', 'original_file_name', 'chunk_index', 'chunk_text', 'chunk_length']
Metadata validation : PASSED

EMBEDDINGS
Shape : (4693, 384)
Dtype : float32
Embedding validation : PASSED

CHROMADB
Collection : wiki_chunks_minilm
Documents  : 4693
ChromaDB v

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2313.97it/s]


Embedding dimension: 384
Embedding model validation : PASSED

PHASE 7 RETRIEVER TEST
Query:
What is artificial intelligence?

Retrieved chunks: 5


,rank,chunk_id,title,category,chunk_index,similarity_score,source_url
0,1,ART_ede7794b399f_chunk_0000,Artificial intelligence,Artificial Intelligence,0,0.780551,https://en.wikipedia.org/wiki/Artificial_intel...
1,2,ART_7b50a3a1d378_chunk_0018,Glossary of artificial intelligence,Artificial Intelligence,18,0.777261,https://en.wikipedia.org/wiki/Glossary_of_arti...
2,3,ART_7b50a3a1d378_chunk_0000,Glossary of artificial intelligence,Artificial Intelligence,0,0.734774,https://en.wikipedia.org/wiki/Glossary_of_arti...
3,4,ART_5b75b0467a43_chunk_0046,Computer science,Data Science,46,0.720271,https://en.wikipedia.org/wiki/Computer_science
4,5,ART_ede7794b399f_chunk_0167,Artificial intelligence,Artificial Intelligence,167,0.714266,https://en.wikipedia.org/wiki/Artificial_intel...



FINAL STATUS
Project configuration : PASSED
Chunk metadata        : PASSED
Embeddings            : PASSED
ChromaDB              : PASSED
Embedding model       : PASSED
Query embedding       : PASSED
Similarity search     : PASSED
Retrieval output      : PASSED

Phase 7 retrieval pipeline successfully loaded.


In [17]:
# ============= BUILD RETRIEVED RAG CONTEXT ===============

# ============================================================
# STEP 1 - CONFIGURATION
# ============================================================

CONTEXT_TOP_K = 5

print("=" * 70)
print("BUILD RETRIEVED CONTEXT")
print("=" * 70)

print("Context top-K:", CONTEXT_TOP_K)


# ============================================================
# STEP 2 - VALIDATE RETRIEVAL RESULTS
# ============================================================

if "test_results_df" not in globals():

    raise RuntimeError(
        "test_results_df was not found. "
        "Please run Phase 9 Cell 2 first."
    )


if test_results_df.empty:

    raise ValueError(
        "Retrieval returned no results."
    )


required_columns = [
    "rank",
    "chunk_id",
    "title",
    "category",
    "source_url",
    "chunk_index",
    "chunk_text",
    "similarity_score"
]

missing_columns = [
    column
    for column in required_columns
    if column not in test_results_df.columns
]

if missing_columns:

    raise ValueError(
        f"Missing required retrieval columns: "
        f"{missing_columns}"
    )


# ============================================================
# STEP 3 - SELECT TOP-K RESULTS
# ============================================================

retrieved_df = (
    test_results_df
    .sort_values(
        by="similarity_score",
        ascending=False
    )
    .head(CONTEXT_TOP_K)
    .reset_index(drop=True)
)


# ============================================================
# STEP 4 - VERIFY RESULTS
# ============================================================

print("\nRetrieved documents:")
print(len(retrieved_df))

print("\nSimilarity scores:")

for _, row in retrieved_df.iterrows():

    print(
        f"Rank {row['rank']}: "
        f"{row['similarity_score']:.4f} - "
        f"{row['title']}"
    )


# ============================================================
# STEP 5 - BUILD FORMATTED CONTEXT
# ============================================================

context_parts = []


for index, row in retrieved_df.iterrows():

    source_number = index + 1

    title = str(
        row["title"]
    ).strip()

    category = str(
        row["category"]
    ).strip()

    source_url = str(
        row["source_url"]
    ).strip()

    chunk_text = str(
        row["chunk_text"]
    ).strip()

    similarity_score = float(
        row["similarity_score"]
    )

    chunk_id = str(
        row["chunk_id"]
    ).strip()

    chunk_index = int(
        row["chunk_index"]
    )


    source_block = f"""
SOURCE {source_number}

Title:
{title}

Category:
{category}

Source URL:
{source_url}

Chunk ID:
{chunk_id}

Chunk Index:
{chunk_index}

Similarity Score:
{similarity_score:.4f}

Content:
{chunk_text}
""".strip()


    context_parts.append(
        source_block
    )


# ============================================================
# STEP 6 - JOIN ALL SOURCES
# ============================================================

retrieved_context = "\n\n" + (
    "\n\n" + ("-" * 70) + "\n\n"
).join(
    context_parts
)


# ============================================================
# STEP 7 - DISPLAY CONTEXT
# ============================================================

print("\n" + "=" * 70)
print("RETRIEVED CONTEXT")
print("=" * 70)

print(retrieved_context)


# ============================================================
# STEP 8 - CONTEXT STATISTICS
# ============================================================

context_character_count = len(
    retrieved_context
)

context_word_count = len(
    retrieved_context.split()
)

print("\n" + "=" * 70)
print("CONTEXT STATISTICS")
print("=" * 70)

print(
    "Number of sources :",
    len(retrieved_df)
)

print(
    "Characters        :",
    context_character_count
)

print(
    "Approx. words     :",
    context_word_count
)


# ============================================================
# STEP 9 - EXTRACT SOURCE INFORMATION
# ============================================================

sources = []


for _, row in retrieved_df.iterrows():

    source = {

        "rank":
            int(row["rank"]),

        "title":
            str(row["title"]).strip(),

        "category":
            str(row["category"]).strip(),

        "source_url":
            str(row["source_url"]).strip(),

        "chunk_id":
            str(row["chunk_id"]).strip(),

        "chunk_index":
            int(row["chunk_index"]),

        "similarity_score":
            float(row["similarity_score"])
    }

    sources.append(source)


# ============================================================
# STEP 10 - DISPLAY SOURCES
# ============================================================

print("\n" + "=" * 70)
print("RETRIEVED SOURCES")
print("=" * 70)

for source in sources:

    print(
        f"\n[{source['rank']}] "
        f"{source['title']}"
    )

    print(
        f"    Category   : "
        f"{source['category']}"
    )

    print(
        f"    Similarity : "
        f"{source['similarity_score']:.4f}"
    )

    print(
        f"    URL        : "
        f"{source['source_url']}"
    )


# ============================================================
# STEP 11 - FINAL VALIDATION
# ============================================================

if not retrieved_context.strip():

    raise ValueError(
        "Retrieved context is empty."
    )


if len(sources) != len(retrieved_df):

    raise ValueError(
        "Source count does not match retrieved documents."
    )


for source in sources:

    if not source["title"]:

        raise ValueError(
            "A retrieved source has an empty title."
        )

    if not source["source_url"]:

        raise ValueError(
            "A retrieved source has an empty URL."
        )


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("FINAL STATUS")
print("=" * 70)

print("Retrieved results      : PASSED")
print("Top-K selection        : PASSED")
print("Context construction   : PASSED")
print("Source extraction      : PASSED")
print("Context validation     : PASSED")

print("\nRetrieved context is ready for Phase 8 prompt generation.")

BUILD RETRIEVED CONTEXT
Context top-K: 5

Retrieved documents:
5

Similarity scores:
Rank 1: 0.7806 - Artificial intelligence
Rank 2: 0.7773 - Glossary of artificial intelligence
Rank 3: 0.7348 - Glossary of artificial intelligence
Rank 4: 0.7203 - Computer science
Rank 5: 0.7143 - Artificial intelligence

RETRIEVED CONTEXT


SOURCE 1

Title:
Artificial intelligence

Category:
Artificial Intelligence

Source URL:
https://en.wikipedia.org/wiki/Artificial_intelligence

Chunk ID:
ART_ede7794b399f_chunk_0000

Chunk Index:
0

Similarity Score:
0.7806

Content:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions 

In [18]:
# ============================================================
# PHASE 9 - CELL 4
# BUILD FINAL RAG PROMPT
# ============================================================

print("=" * 70)
print("BUILD FINAL RAG PROMPT")
print("=" * 70)


# ============================================================
# VALIDATE REQUIRED INPUTS
# ============================================================

if "retrieved_context" not in globals():
    raise RuntimeError(
        "retrieved_context is not available. "
        "Please run Phase 9 Cell 3 first."
    )

if "TEST_QUERY" not in globals():
    raise RuntimeError(
        "TEST_QUERY is not available. "
        "Please run Phase 9 Cell 2 first."
    )


print("✓ Retrieved context found")
print("✓ User query found")


# ============================================================
# PHASE 8 PROMPT BUILDER
# ============================================================

def build_prompt(
    query,
    context,
    prompt_type="question_answering"
):
    """
    Build the final RAG prompt.

    This is the Phase 8 prompt-engineering logic
    used by Phase 9 response generation.
    """

    if not query or not str(query).strip():
        raise ValueError(
            "Query cannot be empty."
        )

    if not context or not str(context).strip():
        raise ValueError(
            "Retrieved context cannot be empty."
        )

    if prompt_type == "question_answering":

        prompt = f"""
You are a knowledgeable AI assistant.

Use the provided context as the primary source for your response.
Answer only using information supported by the provided context.
Do not invent facts or add unsupported information.
If the answer cannot be determined from the provided context,
clearly state that the information is not available in the context.


Task:

Answer the user's question directly and accurately using only the
information supported by the provided context.

Rules:

- Give a concise and complete answer.
- Do not invent facts.
- Do not add unsupported information.
- Do not repeat the same information.
- Do not reproduce the context verbatim unless necessary.
- Do not output source labels such as [Source 1], [Source 2], etc.
- Do not output source URLs.
- Do not reproduce reference lists or external-link sections.
- Do not output "User Input:" or another question/answer section.
- Do not describe these instructions.
- Stop after completing the answer.

If the provided context does not contain enough information to answer
the question, clearly state that the information is not available in
the provided context.


Context:

{context}

Question / Request:
{query}

Response:
""".strip()

    else:

        raise ValueError(
            f"Unsupported prompt type: {prompt_type}"
        )

    return prompt


print("✓ build_prompt() created successfully")


# ============================================================
# BUILD PROMPT
# ============================================================

PROMPT_TYPE = "question_answering"

final_prompt = build_prompt(
    query=TEST_QUERY,
    context=retrieved_context,
    prompt_type=PROMPT_TYPE
)


# ============================================================
# VALIDATE FINAL PROMPT
# ============================================================

if not final_prompt:
    raise RuntimeError(
        "Final prompt generation failed."
    )

if len(final_prompt.strip()) == 0:
    raise RuntimeError(
        "Final prompt is empty."
    )


# ============================================================
# DISPLAY FINAL PROMPT
# ============================================================

print("\n" + "=" * 70)
print("FINAL RAG PROMPT")
print("=" * 70)

print(final_prompt)


# ============================================================
# PROMPT STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("PROMPT STATISTICS")
print("=" * 70)

print(
    "Prompt type        :",
    PROMPT_TYPE
)

print(
    "User query         :",
    TEST_QUERY
)

print(
    "Prompt characters  :",
    len(final_prompt)
)

print(
    "Prompt words       :",
    len(final_prompt.split())
)

print(
    "Retrieved context  :",
    len(retrieved_context)
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("PHASE 9 - CELL 4 STATUS")
print("=" * 70)

print("Prompt builder      : PASSED")
print("Prompt type         : PASSED")
print("User query          : PASSED")
print("Retrieved context   : PASSED")
print("Final prompt        : PASSED")

print(
    "\n✓ Final RAG prompt is ready for Qwen response generation."
)

BUILD FINAL RAG PROMPT
✓ Retrieved context found
✓ User query found
✓ build_prompt() created successfully

FINAL RAG PROMPT
You are a knowledgeable AI assistant.

Use the provided context as the primary source for your response.
Answer only using information supported by the provided context.
Do not invent facts or add unsupported information.
If the answer cannot be determined from the provided context,
clearly state that the information is not available in the context.


Task:

Answer the user's question directly and accurately using only the
information supported by the provided context.

Rules:

- Give a concise and complete answer.
- Do not invent facts.
- Do not add unsupported information.
- Do not repeat the same information.
- Do not reproduce the context verbatim unless necessary.
- Do not output source labels such as [Source 1], [Source 2], etc.
- Do not output source URLs.
- Do not reproduce reference lists or external-link sections.
- Do not output "User Input:" or another

In [19]:
# ============================================================
# PHASE 9 - CELL 6B
# LOAD QWEN GENERATION MODEL 
# ============================================================

import time
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("=" * 70)
print("RELOAD QWEN GENERATION MODEL")
print("=" * 70)

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("\nLoading tokenizer...")

start_time = time.time()

qwen_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_NAME
)

print("✓ Qwen tokenizer loaded")

print("\nLoading Qwen model...")

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

qwen_model = qwen_model.to("cpu")
qwen_model.eval()

load_time = time.time() - start_time

print("✓ Qwen model loaded")

print("\n" + "=" * 70)
print("QWEN MODEL INFORMATION")
print("=" * 70)

parameter_count = sum(
    p.numel()
    for p in qwen_model.parameters()
)

print(
    "Model name       :",
    QWEN_MODEL_NAME
)

print(
    "Parameters       :",
    f"{parameter_count / 1_000_000_000:.2f}B"
)

print(
    "Device           :",
    next(qwen_model.parameters()).device
)

print(
    "Dtype            :",
    next(qwen_model.parameters()).dtype
)

print(
    "Loading time     :",
    f"{load_time:.2f} seconds"
)

print("\n" + "=" * 70)
print("PHASE 9 - CELL 6B STATUS")
print("=" * 70)

print("Qwen tokenizer    : PASSED")
print("Qwen model        : PASSED")
print("CPU configuration : PASSED")

print("\n✓ Qwen generation model is ready.")

RELOAD QWEN GENERATION MODEL

Loading tokenizer...
✓ Qwen tokenizer loaded

Loading Qwen model...


Loading weights: 100%|██████████| 290/290 [00:03<00:00, 88.37it/s] 


✓ Qwen model loaded

QWEN MODEL INFORMATION
Model name       : Qwen/Qwen2.5-0.5B-Instruct
Parameters       : 0.49B
Device           : cpu
Dtype            : torch.float32
Loading time     : 7.95 seconds

PHASE 9 - CELL 6B STATUS
Qwen tokenizer    : PASSED
Qwen model        : PASSED
CPU configuration : PASSED

✓ Qwen generation model is ready.


In [20]:
# ============================================================
# PHASE 9 - CELL 7
# GENERATE FINAL RAG ANSWER USING QWEN
# ============================================================

import time
import torch

print("=" * 70)
print("PHASE 9 - CELL 7: QWEN RAG RESPONSE GENERATION")
print("=" * 70)


# ------------------------------------------------------------
# VALIDATE VARIABLES
# ------------------------------------------------------------

if "qwen_model" not in globals():
    raise RuntimeError(
        "qwen_model not found. Please run Cell 6B first."
    )

if "qwen_tokenizer" not in globals():
    raise RuntimeError(
        "qwen_tokenizer not found. Please run Cell 6B first."
    )

if "final_prompt" not in globals():
    raise RuntimeError(
        "final_prompt not found. Please run Cell 4 first."
    )


print("✓ Qwen model found")
print("✓ Qwen tokenizer found")
print("✓ final_prompt found")


# ------------------------------------------------------------
# CREATE CHAT MESSAGES
# ------------------------------------------------------------

messages = [
    {
        "role": "system",
        "content": (
            "You are a knowledgeable AI assistant. "
            "Answer only using information supported by "
            "the provided context. Do not invent facts."
        )
    },
    {
        "role": "user",
        "content": final_prompt
    }
]


# ------------------------------------------------------------
# TOKENIZE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("PREPARING MODEL INPUT")
print("-" * 70)

inputs = qwen_tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
)

inputs = {
    key: value.to("cpu")
    for key, value in inputs.items()
}

input_tokens = inputs["input_ids"].shape[1]

print("✓ Input prepared")
print("Input tokens:", input_tokens)


# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("GENERATING FINAL RAG ANSWER")
print("-" * 70)

print("Model : Qwen/Qwen2.5-0.5B-Instruct")
print("Device: CPU")
print("Please wait...")

start_time = time.time()

with torch.no_grad():

    outputs = qwen_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.05,
        eos_token_id=qwen_tokenizer.eos_token_id,
        pad_token_id=qwen_tokenizer.eos_token_id
    )

generation_time = time.time() - start_time


# ------------------------------------------------------------
# EXTRACT GENERATED TOKENS
# ------------------------------------------------------------

generated_tokens = outputs[
    0,
    input_tokens:
]


# ------------------------------------------------------------
# DECODE
# ------------------------------------------------------------

rag_answer = qwen_tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()


# ------------------------------------------------------------
# VALIDATE
# ------------------------------------------------------------

if not rag_answer:

    raise RuntimeError(
        "Qwen generated an empty response."
    )


# ------------------------------------------------------------
# DISPLAY ANSWER
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL RAG ANSWER")
print("=" * 70)

print(rag_answer)


# ------------------------------------------------------------
# STATISTICS
# ------------------------------------------------------------

output_tokens = len(generated_tokens)

print("\n" + "=" * 70)
print("GENERATION STATISTICS")
print("=" * 70)

print("Input tokens      :", input_tokens)
print("Output tokens     :", output_tokens)
print(
    "Generation time   :",
    f"{generation_time:.2f} seconds"
)

if generation_time > 0:

    print(
        "Tokens / second   :",
        f"{output_tokens / generation_time:.2f}"
    )


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 9 - CELL 7 STATUS")
print("=" * 70)

print("Prompt preparation : PASSED")
print("Qwen generation    : PASSED")
print("Answer extraction  : PASSED")
print("Answer validation  : PASSED")

print("\n✓ FINAL RAG ANSWER GENERATED SUCCESSFULLY.")

PHASE 9 - CELL 7: QWEN RAG RESPONSE GENERATION
✓ Qwen model found
✓ Qwen tokenizer found
✓ final_prompt found

----------------------------------------------------------------------
PREPARING MODEL INPUT
----------------------------------------------------------------------
✓ Input prepared
Input tokens: 1170

----------------------------------------------------------------------
GENERATING FINAL RAG ANSWER
----------------------------------------------------------------------
Model : Qwen/Qwen2.5-0.5B-Instruct
Device: CPU
Please wait...

FINAL RAG ANSWER
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions 

In [21]:
# ============================================================
# PHASE 9 - CELL 8
# FINAL RAG RESPONSE
# ============================================================

print("=" * 70)
print("PHASE 9 - CELL 8: FINAL RAG RESPONSE")
print("=" * 70)


# ------------------------------------------------------------
# VALIDATE REQUIRED VARIABLES
# ------------------------------------------------------------

required_variables = [
    "TEST_QUERY",
    "rag_answer",
    "retrieved_context"
]

for variable in required_variables:

    if variable not in globals():

        raise RuntimeError(
            f"Required variable '{variable}' is not available."
        )

print("✓ Required variables available")


# ------------------------------------------------------------
# EXTRACT RETRIEVED DOCUMENT INFORMATION
# ------------------------------------------------------------

sources = []

# Try to use the retrieval dataframe if available
if "retrieved_df" in globals():

    for _, row in retrieved_df.iterrows():

        source = {
            "title": row.get("title", ""),
            "category": row.get("category", ""),
            "source_url": row.get("source_url", ""),
            "similarity_score": float(
                row.get("similarity_score", 0)
            )
        }

        sources.append(source)


# ------------------------------------------------------------
# FALLBACK: CHECK COMMON RETRIEVAL VARIABLE NAMES
# ------------------------------------------------------------

elif "results_df" in globals():

    for _, row in results_df.iterrows():

        source = {
            "title": row.get("title", ""),
            "category": row.get("category", ""),
            "source_url": row.get("source_url", ""),
            "similarity_score": float(
                row.get("similarity_score", 0)
            )
        }

        sources.append(source)


# ------------------------------------------------------------
# CONFIDENCE CALCULATION
# ------------------------------------------------------------

if sources:

    similarity_scores = [
        s["similarity_score"]
        for s in sources
        if s["similarity_score"] is not None
    ]

    if similarity_scores:

        top_score = max(similarity_scores)

        # Keep confidence within 0-1
        confidence_score = max(
            0.0,
            min(1.0, top_score)
        )

    else:

        confidence_score = 0.0

else:

    confidence_score = 0.0


# ------------------------------------------------------------
# DISPLAY FINAL ANSWER
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("USER QUESTION")
print("=" * 70)

print(TEST_QUERY)


print("\n" + "=" * 70)
print("FINAL ANSWER")
print("=" * 70)

print(rag_answer)


# ------------------------------------------------------------
# DISPLAY SOURCES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RETRIEVED SOURCES")
print("=" * 70)

if sources:

    for index, source in enumerate(
        sources,
        start=1
    ):

        print(
            f"\n[{index}] {source['title']}"
        )

        print(
            "Category :",
            source["category"]
        )

        print(
            "Similarity:",
            f"{source['similarity_score']:.4f}"
        )

        print(
            "URL      :",
            source["source_url"]
        )

else:

    print("No source metadata available.")


# ------------------------------------------------------------
# CONFIDENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONFIDENCE")
print("=" * 70)

print(
    f"Confidence score: {confidence_score:.4f}"
)

print(
    f"Confidence percentage: "
    f"{confidence_score * 100:.2f}%"
)


# ------------------------------------------------------------
# SAVE FINAL RESPONSE OBJECT
# ------------------------------------------------------------

final_rag_response = {
    "question": TEST_QUERY,
    "answer": rag_answer,
    "sources": sources,
    "confidence_score": confidence_score
}


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PHASE 9 - CELL 8 STATUS")
print("=" * 70)

print("Question           : PASSED")
print("Generated answer   : PASSED")
print("Source extraction  : PASSED")
print("Confidence         : PASSED")
print("Final response     : PASSED")

print(
    "\n✓ Phase 9 response generation completed."
)

PHASE 9 - CELL 8: FINAL RAG RESPONSE
✓ Required variables available

USER QUESTION
What is artificial intelligence?

FINAL ANSWER
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.

RETRIEVED SOURCES

[1] Artificial intelligence
Category : Artificial Intelligence
Similarity: 0.7806
URL      : https://en.wikipedia.org/wiki/Artificial_intelligence

[2] Glossary of artificial intelligence
Category : Artificial Intelligence
Similarity: 0.7773
URL      : https://en.wikipedia.org/wiki/Glossary_of_artificial_intelligence

[3] Glossary of artificial intellige